### Bibliotecas

Importando as bibliotecas.

In [4070]:
import pandas as pd
import psycopg
import numpy as np

### Configurações

Configurando o acesso à API.

In [4071]:
api_key = "cmv5ao0pn09jztzbfaienbbri"

Determinando o campeonato e as temporadas que irei acessar.

O formato é "país", "campeonato", "temporada".

In [4072]:
bases = [
    ("netherlands", "knvb-beker", "2026-2027"),
    ("netherlands", "knvb-beker", "2025-2026"),
    ("netherlands", "knvb-beker", "2024-2025"),
    ("netherlands", "knvb-beker", "2023-2024"),
    ("netherlands", "knvb-beker", "2022-2023"),
    ("netherlands", "knvb-beker", "2021-2022"),
]

Setup da coleta de dados.

In [4073]:
pais_campeonato = 'Holanda'
campeonato = 'Holandês'
divisao = 1
continente = 'Europa'
formato = 'Liga'

Apontando para o link de acesso à informação.

In [4074]:
BASE_URL = "https://futpythontrader.com.br/api/download"

Criando o DataFrame.

In [4075]:
dataframes = []
for pais, liga, temporada in bases:
    url = f"{BASE_URL}/{pais}/{liga}/{temporada}?api_key={api_key}"
    df = pd.read_csv(url)
    dataframes.append(df)
    print(f"{pais}/{liga}/{temporada}: {len(df)} partidas")

HTTPError: HTTP Error 404: Not Found

Exibindo as primeiras linhas da base de dados.

In [4047]:
df.head(3)

,Match_ID,Country,Season,Div,League,Date,Time,Round,Home,Away,...,Yellow_Cards_Home_2T,Yellow_Cards_Away_2T,Red_Cards_Home_2T,Red_Cards_Away_2T,Fouls_Home_2T,Fouls_Away_2T,Offsides_Home_2T,Offsides_Away_2T,Goalkeeper_Saves_Home_2T,Goalkeeper_Saves_Away_2T
0,xUgqKyBa,NETHERLANDS,2021-2022,Eredivisie 2021 2022,NETHERLANDS 1,29/05/2022,13:00,Relegation,Den Haag,Excelsior,...,2.0,0.0,1.0,1.0,5.0,4.0,2.0,2.0,2.0,1.0
1,zPVi0NcL,NETHERLANDS,2021-2022,Eredivisie 2021 2022,NETHERLANDS 1,29/05/2022,09:30,Conference League,AZ Alkmaar,Vitesse,...,0.0,2.0,NaN,NaN,4.0,4.0,1.0,0.0,4.0,3.0
2,MLRm13rF,NETHERLANDS,2021-2022,Eredivisie 2021 2022,NETHERLANDS 1,26/05/2022,15:00,Conference League,Vitesse,AZ Alkmaar,...,1.0,0.0,NaN,NaN,6.0,5.0,0.0,0.0,2.0,1.0


### Tratamento de dados

Nessa etapa temos alguns objetivos:

1. Verificar se a base de dados tem valores duplicados e/ou nulos;
2. Escolher as colunas que eu vou trabalhar;
3. Renomear as colunas para o português;
4. Verificar se todas as colunas estão no formato correto.

#### 1. Verificando valores nulos e duplicados

Verificando os valores nulos.

In [4048]:
df.isnull().sum()

Match_ID                    0
Country                     0
Season                      0
Div                         0
League                      0
                           ..
Fouls_Away_2T               0
Offsides_Home_2T            0
Offsides_Away_2T            0
Goalkeeper_Saves_Home_2T    0
Goalkeeper_Saves_Away_2T    0
Length: 261, dtype: int64

Verificando os valores duplicados.

In [4049]:
df.duplicated().sum()

np.int64(0)

#### 2. Colunas de trabalho

Para identificar as colunas de trabalho, baixei o CSV para visualizar com mais calma e detalhe a informação de todas as colunas.

In [4050]:

# Colunas que serão mantidas
colunas = """
Match_ID Country Season Div League Date Time Round Home Away
Home_Score Away_Score Min_Goals_Home Min_Goals_Away
Bookie_1X2_HT Odd_1_HT Odd_X_HT Odd_2_HT
Bookie_1X2_FT Odd_1_FT Odd_X_FT Odd_2_FT
Bookie_OU_HT_0_5 Over_HT_0_5 Under_HT_0_5
Bookie_OU_HT_1_5 Over_HT_1_5 Under_HT_1_5
Bookie_OU_HT_2_5 Over_HT_2_5 Under_HT_2_5
Bookie_OU_FT_0_5 Over_FT_0_5 Under_FT_0_5
Bookie_OU_FT_1_5 Over_FT_1_5 Under_FT_1_5
Bookie_OU_FT_2_5 Over_FT_2_5 Under_FT_2_5
Bookie_OU_FT_3_5 Over_FT_3_5 Under_FT_3_5
Bookie_OU_FT_4_5 Over_FT_4_5 Under_FT_4_5
Bookie_BTTS BTTS_Yes BTTS_No
Bookie_DC DC_1X DC_12 DC_X2
Bookie_CS_0_0 CS_0_0
Bookie_CS_0_1 CS_0_1
Bookie_CS_0_2 CS_0_2
Bookie_CS_0_3 CS_0_3
Bookie_CS_1_0 CS_1_0
Bookie_CS_1_1 CS_1_1
Bookie_CS_1_2 CS_1_2
Bookie_CS_1_3 CS_1_3
Bookie_CS_2_0 CS_2_0
Bookie_CS_2_1 CS_2_1
Bookie_CS_2_2 CS_2_2
Bookie_CS_2_3 CS_2_3
Bookie_CS_3_0 CS_3_0
Bookie_CS_3_1 CS_3_1
Bookie_CS_3_2 CS_3_2
Bookie_CS_3_3 CS_3_3
""".split()

# Verificar se alguma coluna está ausente
faltantes = [c for c in colunas if c not in df.columns]

if faltantes:
    raise ValueError(f"Colunas não encontradas: {faltantes}")

# Manter apenas as colunas selecionadas
df = df.loc[:, colunas].copy()

Eliminando as colunas que retornam o nome da casa de apostas.

In [4051]:
# Remover todas as colunas que começam com "Bookie_"
df = df.loc[
    :, ~df.columns.str.startswith("Bookie_", "Time")
].copy()

C:\Users\stefa\AppData\Local\Temp\ipykernel_20084\1889842315.py:3: FutureWarning: Allowing a non-bool 'na' in obj.str.startswith is deprecated and will raise in a future version.
  :, ~df.columns.str.startswith("Bookie_", "Time")


Verificando se, entre as colunas mantidas, há algum valor nulo.

In [4052]:
df.isnull().sum().reset_index(name='nulos').query('nulos > 0')

,index,nulos


#### 3. Renomeando as colunas

In [4053]:
# Renomear colunas
df = df.rename(columns={
    "Match_ID": "jogo_id",
    "Country": "pais",
    "Season": "temporada",
    "Div": "divisao",
    "League": "campeonato",
    "Date": "data",
    "Round": "rodada",
    "Home": "casa",
    "Away": "fora",
    "Home_Score": "gols_casa",
    "Away_Score": "gols_fora",
    "Min_Goals_Home": "minutos_gols_casa",
    "Min_Goals_Away": "minutos_gols_fora",

    # Match Odds
    "Odd_1_HT": "odd_casa_ht",
    "Odd_X_HT": "odd_empate_ht",
    "Odd_2_HT": "odd_fora_ht",
    "Odd_1_FT": "odd_casa_ft",
    "Odd_X_FT": "odd_empate_ft",
    "Odd_2_FT": "odd_fora_ft",

    # Over / Under - Primeiro tempo
    "Over_HT_0_5": "over_0_5_ht",
    "Under_HT_0_5": "under_0_5_ht",
    "Over_HT_1_5": "over_1_5_ht",
    "Under_HT_1_5": "under_1_5_ht",
    "Over_HT_2_5": "over_2_5_ht",
    "Under_HT_2_5": "under_2_5_ht",

    # Over / Under - Jogo completo
    "Over_FT_0_5": "over_0_5_ft",
    "Under_FT_0_5": "under_0_5_ft",
    "Over_FT_1_5": "over_1_5_ft",
    "Under_FT_1_5": "under_1_5_ft",
    "Over_FT_2_5": "over_2_5_ft",
    "Under_FT_2_5": "under_2_5_ft",
    "Over_FT_3_5": "over_3_5_ft",
    "Under_FT_3_5": "under_3_5_ft",
    "Over_FT_4_5": "over_4_5_ft",
    "Under_FT_4_5": "under_4_5_ft",

    # Ambos marcam
    "BTTS_Yes": "ambos_marcam_sim",
    "BTTS_No": "ambos_marcam_nao",

    # Dupla chance
    "DC_1X": "dupla_chance_1x",
    "DC_12": "dupla_chance_12",
    "DC_X2": "dupla_chance_x2",

    # Placar correto
    "CS_0_0": "correct_score_0_0",
    "CS_0_1": "correct_score_0_1",
    "CS_0_2": "correct_score_0_2",
    "CS_0_3": "correct_score_0_3",
    "CS_1_0": "correct_score_1_0",
    "CS_1_1": "correct_score_1_1",
    "CS_1_2": "correct_score_1_2",
    "CS_1_3": "correct_score_1_3",
    "CS_2_0": "correct_score_2_0",
    "CS_2_1": "correct_score_2_1",
    "CS_2_2": "correct_score_2_2",
    "CS_2_3": "correct_score_2_3",
    "CS_3_0": "correct_score_3_0",
    "CS_3_1": "correct_score_3_1",
    "CS_3_2": "correct_score_3_2",
    "CS_3_3": "correct_score_3_3"
})

Eliminando a coluna com o horário do jogo.

In [4054]:
df = df.drop(columns='Time')

Mapeando as informações de forma correta.

In [4055]:
df['pais'] = pais_campeonato
df['campeonato'] = campeonato
df['divisao'] = divisao
df['continente'] = continente
df['formato'] = formato

Alterando a posição das colunas "continente" e "formato".

In [4056]:
colunas = df.columns.tolist()

colunas.remove('continente')
colunas.insert(colunas.index('pais') + 1, 'continente')

df = df[colunas]

In [4057]:
colunas = df.columns.tolist()

colunas.remove('formato')
colunas.insert(colunas.index('divisao') + 1, 'formato')

df = df[colunas]

Corrigindo o nome das rodadas

In [4058]:
df['rodada'] = df['rodada'].str.replace('Round ', '', regex=False)

Verificando os valores únicos da coluna "rodada".

In [4059]:
df.rodada.unique()

array(['Relegation', 'Conference League', '34', '33', '32', '31', '30',
       '25', '23', '29', '28', '27', '26', '24', '22', '21', '20', '19',
       '18', '17', '16', '15', '2', '4', '14', '13', '12', '11', '10',
       '9', '8', '7', '6', '5', '3', '1'], dtype=object)

#### Verificando se as colunas estão corretas

Ajustando o nome das rodadas não numéricas.

In [4060]:
df['rodada'] = df['rodada'].str.replace('Promotion', 'Fase final')
df['rodada'] = df['rodada'].str.replace('Super final', 'Final')
df['rodada'] = df['rodada'].str.replace('Play Offs', 'Playoff')
df['rodada'] = df['rodada'].str.replace('Conference League', 'Fase 1')
df['rodada'] = df['rodada'].str.replace('Relegation', 'Fase 1')
df['rodada'] = df['rodada'].str.replace('Championship Group', 'Fase 1')
df['rodada'] = df['rodada'].str.replace('Relegation Group', 'Fase 1')
df['rodada'] = df['rodada'].str.replace('Fase 1 Group', 'Fase 1')
df['rodada'] = df['rodada'].str.replace('First stage', 'Fase 1')
df['rodada'] = df['rodada'].str.replace('Second stage', 'Fase 2')
df['rodada'] = df['rodada'].str.replace('Grand Final', 'Final')
df['rodada'] = df['rodada'].str.replace('7th-10th places', 'Fase 2')
df['rodada'] = df['rodada'].str.replace('Fase final Group', 'Fase final')
df['rodada'] = df['rodada'].str.replace('Group Stage', 'Fase de grupos')
df['rodada'] = df['rodada'].str.replace('Qualification', 'Fase 2')
df['rodada'] = df['rodada'].str.replace('Preliminary', 'Fase 1')
df['rodada'] = df['rodada'].str.replace('Play Out', 'Playoff')
df['rodada'] = df['rodada'].str.replace('Group A', 'Fase de grupos')
df['rodada'] = df['rodada'].str.replace('Group B', 'Fase de grupos')
df['rodada'] = df['rodada'].str.replace('Group C', 'Fase de grupos')
df['rodada'] = df['rodada'].str.replace('Group D', 'Fase de grupos')

In [4061]:
df.rodada.unique()

array(['Fase 1', '34', '33', '32', '31', '30', '25', '23', '29', '28',
       '27', '26', '24', '22', '21', '20', '19', '18', '17', '16', '15',
       '2', '4', '14', '13', '12', '11', '10', '9', '8', '7', '6', '5',
       '3', '1'], dtype=object)

### Configurando o acesso ao banco de dados

#### Credenciais de acesso

In [4062]:
DB_CONFIG = {
    "host": "aws-0-sa-east-1.pooler.supabase.com",
    "port": 5432,
    "dbname": "postgres",
    "user": "postgres.hzwdcminwjvheuddxzjq",
    "password": "Skt8punk@stefano",
    "sslmode": "require",
    "connect_timeout": 30
}

TAMANHO_LOTE = 1000

#### Colunas na ordem do banco

In [4063]:
COLUNAS = """
data
temporada
pais
continente
campeonato
divisao
formato
jogo_id
rodada
casa
gols_casa
gols_fora
fora
minutos_gols_casa
minutos_gols_fora
odd_casa_ht
odd_empate_ht
odd_fora_ht
odd_casa_ft
odd_empate_ft
odd_fora_ft
over_0_5_ht
under_0_5_ht
over_1_5_ht
under_1_5_ht
over_2_5_ht
under_2_5_ht
over_0_5_ft
under_0_5_ft
over_1_5_ft
under_1_5_ft
over_2_5_ft
under_2_5_ft
over_3_5_ft
under_3_5_ft
over_4_5_ft
under_4_5_ft
ambos_marcam_sim
ambos_marcam_nao
dupla_chance_1x
dupla_chance_12
dupla_chance_x2
correct_score_0_0
correct_score_0_1
correct_score_0_2
correct_score_0_3
correct_score_1_0
correct_score_1_1
correct_score_1_2
correct_score_1_3
correct_score_2_0
correct_score_2_1
correct_score_2_2
correct_score_2_3
correct_score_3_0
correct_score_3_1
correct_score_3_2
correct_score_3_3
""".split()

#### Carregando e preparando o DataFrame

In [4064]:
def preparar_dataframe(df):

    print("[1/3] Preparando DataFrame...")

    dados = df.copy()

    # Garantir coluna formato
    if "formato" not in dados.columns:
        dados["formato"] = None

    # Validar colunas
    faltantes = [
        c for c in COLUNAS
        if c not in dados.columns
    ]

    if faltantes:
        raise ValueError(
            f"Colunas ausentes: {faltantes}"
        )

    # Organizar colunas
    dados = dados[COLUNAS].copy()

    # Converter datas
    dados["data"] = pd.to_datetime(
        dados["data"],
        dayfirst=True,
        errors="raise"
    ).dt.date

    if dados["data"].isna().any():
        raise ValueError("Existem partidas sem data.")

    # Converter temporada para texto
    dados["temporada"] = (
        dados["temporada"].astype("string")
    )

    # Converter campos inteiros
    for coluna in [
        "divisao",
        "gols_casa",
        "gols_fora"
    ]:
        dados[coluna] = pd.to_numeric(
            dados[coluna],
            errors="raise"
        ).astype("Int64")

    # Converter colunas de odds
    inicio = COLUNAS.index("odd_casa_ht")

    for coluna in COLUNAS[inicio:]:
        dados[coluna] = pd.to_numeric(
            dados[coluna],
            errors="raise"
        )

    # Validar jogo_id
    dados["jogo_id"] = (
        dados["jogo_id"].astype("string").str.strip()
    )

    if (
        dados["jogo_id"].isna().any()
        or dados["jogo_id"].eq("").any()
    ):
        raise ValueError("Existem jogos sem jogo_id.")

    # Remover IDs duplicados no lote
    duplicados = dados["jogo_id"].duplicated().sum()

    if duplicados:
        print(
            f"[AVISO] {duplicados} jogos duplicados."
        )

        dados = dados.drop_duplicates(
            subset="jogo_id",
            keep="last"
        )

    print(f"[OK] {len(dados):,} jogos preparados.")

    return dados

#### Convertendo os valores para o script

In [4065]:
def converter_valor(valor):

    if isinstance(valor, np.generic):
        valor = valor.item()

    if isinstance(valor, (list, tuple)):
        return str(valor)

    if pd.isna(valor):
        return None

    return valor

#### Inserindo as informações no banco de dados

In [4066]:
def alimentar_odds(df):

    dados = preparar_dataframe(df)

    print("[2/3] Conectando ao Supabase...")

    # Montar SQL
    colunas_sql = ", ".join(COLUNAS)

    placeholders = ", ".join(
        ["%s"] * len(COLUNAS)
    )

    atualizacoes = []

    for coluna in COLUNAS:

        if coluna == "jogo_id":
            continue

        if coluna == "formato":
            atualizacoes.append(
                "formato = COALESCE("
                "EXCLUDED.formato, odds.formato)"
            )
        else:
            atualizacoes.append(
                f"{coluna} = EXCLUDED.{coluna}"
            )

    sql = f"""
        INSERT INTO public.odds (
            {colunas_sql}
        )
        VALUES (
            {placeholders}
        )
        ON CONFLICT (jogo_id)
        DO UPDATE SET
            {", ".join(atualizacoes)}
    """

    # Preparar registros
    registros = [
        tuple(converter_valor(v) for v in linha)
        for linha in dados.itertuples(
            index=False,
            name=None
        )
    ]

    total = len(registros)

    if total == 0:
        print("[AVISO] Nenhum registro para importar.")
        return

    print("[3/3] Enviando registros...")

    with psycopg.connect(**DB_CONFIG) as conexao:

        with conexao.cursor() as cursor:

            for inicio in range(
                0,
                total,
                TAMANHO_LOTE
            ):

                lote = registros[
                    inicio:inicio + TAMANHO_LOTE
                ]

                cursor.executemany(sql, lote)

                enviados = inicio + len(lote)

                print(
                    f"[OK] {enviados:,}/{total:,}"
                    " registros processados."
                )

            cursor.execute(
                "SELECT COUNT(*) FROM public.odds"
            )

            total_banco = cursor.fetchone()[0]

        conexao.commit()

    print("\nIMPORTAÇÃO CONCLUÍDA!")
    print(f"Registros processados: {total:,}")
    print(f"Total na tabela odds: {total_banco:,}")

#### Executando

In [4067]:
alimentar_odds(df)

[1/3] Preparando DataFrame...
[OK] 324 jogos preparados.
[2/3] Conectando ao Supabase...
[3/3] Enviando registros...
[OK] 324/324 registros processados.

IMPORTAÇÃO CONCLUÍDA!
Registros processados: 324
Total na tabela odds: 17,547


### Validando a consulta no banco de dados

In [4068]:

import pandas as pd
import psycopg

# País que deseja consultar
pais = "Belga"

with psycopg.connect(**DB_CONFIG) as conexao:
    with conexao.cursor() as cursor:

        # 1. Quantidade de jogos do país
        cursor.execute("""
            SELECT COUNT(*)
            FROM public.odds
            WHERE pais = %s
        """, (pais,))

        total = cursor.fetchone()[0]

        print(f"Total de jogos de {pais}: {total:,}")

        # 2. Consultar os últimos 20 registros
        cursor.execute("""
            SELECT *
            FROM public.odds
            WHERE pais = %s
            ORDER BY id DESC
            LIMIT 20
        """, (pais,))

        colunas = [col.name for col in cursor.description]
        registros = cursor.fetchall()

        df_consulta = pd.DataFrame(
            registros,
            columns=colunas
        )

        # 3. Conferir jogos do DataFrame
        df_pais = df.loc[df["pais"] == pais]

        ids_df = set(
            df_pais["jogo_id"].dropna().astype(str).str.strip()
        )

        ids_banco = set()
        lista_ids = list(ids_df)

        for inicio in range(0, len(lista_ids), 1000):
            lote = lista_ids[inicio:inicio + 1000]

            cursor.execute("""
                SELECT jogo_id
                FROM public.odds
                WHERE pais = %s
                  AND jogo_id = ANY(%s)
            """, (pais, lote))

            ids_banco.update(
                str(linha[0]) for linha in cursor.fetchall()
            )

        faltantes = ids_df - ids_banco

print(f"\nJogos no DataFrame: {len(ids_df):,}")
print(f"Jogos encontrados no banco: {len(ids_banco):,}")
print(f"Jogos não encontrados: {len(faltantes):,}")

if not faltantes:
    print(f"\n[OK] Todos os jogos de {pais} estão no Supabase!")
else:
    print("\n[ATENÇÃO] Existem jogos não encontrados.")
    print("Exemplos:", list(faltantes)[:10])

display(df_consulta)


Total de jogos de Belga: 0

Jogos no DataFrame: 0
Jogos encontrados no banco: 0
Jogos não encontrados: 0

[OK] Todos os jogos de Belga estão no Supabase!


,id,data,temporada,pais,continente,campeonato,divisao,formato,jogo_id,rodada,...,correct_score_1_3,correct_score_2_0,correct_score_2_1,correct_score_2_2,correct_score_2_3,correct_score_3_0,correct_score_3_1,correct_score_3_2,correct_score_3_3,criado_em


In [4069]:
import winsound

# winsound.Beep(frequencia, duracao_ms)
# Exemplo: Som de 1000 Hz por 500 milissegundos (meio segundo)
winsound.Beep(1000, 500)
